[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Maxencegu/python-datascience-m1/blob/main/cours/cm05_modules_packages_apis_et_donnees/cm05_cours.ipynb)

# **CM05 — Modules, Packages, APIs & Données**

**Python & Data Science · UPJV Amiens · M1 Économie**

---

Python tire sa puissance de son écosystème : des milliers de bibliothèques couvrent tout, de l'analyse de données à l'intelligence artificielle. Ce CM vous apprend à les utiliser.

À l'issue de ce CM, vous saurez :
- distinguer modules, packages et fonctions built-in ;
- importer du code Python (4 syntaxes) ;
- utiliser les modules de la bibliothèque standard ;
- installer et gérer des paquets avec `uv` ;
- interroger une API web avec `requests` ;
- analyser et visualiser des données avec Pandas et Matplotlib ;
- lire et écrire des fichiers CSV, JSON et Excel.

---
## 1. Modules et packages

### 1.1 Trois niveaux d'organisation du code

| Niveau | Description | Exemple |
|--------|-------------|------|
| **Fonction** | Bloc de code réutilisable | `len()`, `sum()` |
| **Module** | Fichier `.py` regroupant des fonctions liées | `math`, `random`, `os` |
| **Package** | Dossier de modules | `pandas`, `matplotlib`, `numpy` |

### 1.2 Les quatre façons d'importer

In [ ]:
import math                        # Importer le module entier
print(math.pi)                     # 3.141592...
print(math.sqrt(16))               # 4.0
print(math.floor(3.7))             # 3

In [ ]:
from math import sqrt, pi          # Importer des éléments spécifiques
print(sqrt(25))                    # 5.0  — pas besoin de math.
print(pi)                          # 3.141592...

In [ ]:
import numpy as np                 # Alias — convention pour les paquets courants
import pandas as pd
import matplotlib.pyplot as plt

# Les alias np, pd, plt sont des standards universels
tableau = np.array([1, 2, 3, 4, 5])
print(tableau.mean())              # 3.0

In [ ]:
# À éviter : from module import *
# from math import *   ← pollue l'espace de noms, conflits potentiels

# Préférer toujours les trois formes précédentes

---
## 2. Bibliothèque standard

Python inclut des dizaines de modules prêts à l'emploi, sans installation.

In [ ]:
# random — nombres aléatoires
import random

print(random.random())           # float entre 0 et 1
print(random.randint(1, 6))      # entier entre 1 et 6 (dé)
print(random.choice(["Pile", "Face"]))  # tirage aléatoire

fruits = ["pomme", "banane", "cerise", "datte"]
random.shuffle(fruits)           # mélange en place
print(fruits)

In [ ]:
# datetime — dates et heures
from datetime import datetime, timedelta

maintenant = datetime.now()
print(f"Date et heure : {maintenant.strftime('%d/%m/%Y %H:%M')}")
print(f"Année : {maintenant.year}")

dans_30_jours = maintenant + timedelta(days=30)
print(f"Dans 30 jours : {dans_30_jours.strftime('%d/%m/%Y')}")

In [ ]:
# os — interaction avec le système de fichiers
import os

print(os.getcwd())               # répertoire courant
print(os.path.exists("/content"))  # True sur Colab

# Construire des chemins de façon portable (Windows/Mac/Linux)
chemin = os.path.join("données", "ventes", "2024.csv")
print(chemin)

In [ ]:
# json — sérialisation/désérialisation JSON
import json

# Dictionnaire Python → chaîne JSON
produit = {"nom": "Laptop", "prix": 999.99, "disponible": True}
chaine_json = json.dumps(produit, ensure_ascii=False, indent=2)
print(chaine_json)

# Chaîne JSON → dictionnaire Python
data = json.loads('{"ville": "Amiens", "population": 133000}')
print(data["ville"])

---
## 3. Installer des paquets avec uv

```bash
# Commandes uv essentielles (dans Git Bash)
uv add requests               # ajouter un paquet au projet
uv add pandas matplotlib      # plusieurs paquets d'un coup
uv add requests==2.31.0       # version spécifique
uv add requests               # mise à jour : uv choisit la version compatible
uv remove requests            # supprimer un paquet
```

### 3.1 `pyproject.toml` et `uv.lock` — dépendances du projet

`uv add` met à jour automatiquement `pyproject.toml` (liste des dépendances) et `uv.lock` (versions exactes verrouillées).

```bash
# Sur une autre machine, recréer l'environnement identique :
uv sync
```

Exemple de `pyproject.toml` après `uv add pandas matplotlib requests openpyxl` :
```toml
[project]
name = "mon-projet"
requires-python = ">=3.12"
dependencies = [
    "pandas>=2.2.0",
    "matplotlib>=3.8.2",
    "requests>=2.31.0",
    "openpyxl>=3.1.2",
]
```

### 3.2 Trouver des paquets

- **PyPI** (pypi.org) — dépôt officiel, 500 000+ paquets
- Recherche : directement sur pypi.org (`pip search` a été retiré)
- Critères de choix : étoiles GitHub, date de mise à jour, documentation

---
## 4. APIs web avec `requests`

### 4.1 Qu'est-ce qu'une API ?

Une **API** (Application Programming Interface) est un service en ligne qui expose des données ou des fonctionnalités via des URLs. Votre programme envoie une requête HTTP → l'API répond avec des données (souvent en JSON).

```
Votre programme  ──── requête GET ────►  Serveur API
                 ◄─── réponse JSON ────  (météo, données économiques, etc.)
```

In [ ]:
import requests

# API Open-Meteo — météo gratuite, sans clé d'authentification
# Documentation : https://open-meteo.com

url = "https://api.open-meteo.com/v1/forecast"
parametres = {
    "latitude": 49.90,    # Amiens
    "longitude": 2.30,
    "current": "temperature_2m,wind_speed_10m",
    "hourly": "temperature_2m,precipitation_probability"
}

reponse = requests.get(url, params=parametres)
print(f"Statut HTTP : {reponse.status_code}")  # 200 = succès

In [ ]:
# Extraire les données JSON
data = reponse.json()

temperature = data["current"]["temperature_2m"]
vent = data["current"]["wind_speed_10m"]

print(f"Température à Amiens : {temperature}°C")
print(f"Vent : {vent} km/h")

In [ ]:
# Encapsuler dans une fonction réutilisable
def obtenir_meteo(latitude, longitude):
    """Retourne la météo actuelle pour des coordonnées GPS."""
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "current": "temperature_2m,wind_speed_10m,precipitation"
    }
    reponse = requests.get(url, params=params)
    if reponse.status_code == 200:
        return reponse.json()["current"]
    return None

# Comparer plusieurs villes
villes = {
    "Amiens":  (49.90, 2.30),
    "Paris":   (48.85, 2.35),
    "Marseille": (43.30, 5.38),
}

print(f"{'Ville':<12} {'Température':>12} {'Vent':>10}")
print("-" * 36)
for ville, (lat, lon) in villes.items():
    meteo = obtenir_meteo(lat, lon)
    if meteo:
        print(f"{ville:<12} {meteo['temperature_2m']:>10.1f}°C {meteo['wind_speed_10m']:>8.1f} km/h")

### 4.2 Codes de statut HTTP

| Code | Signification |
|------|---------------|
| **200** | Succès |
| **400** | Requête invalide (vos paramètres sont incorrects) |
| **401** | Non autorisé (clé API manquante ou invalide) |
| **404** | Ressource introuvable |
| **429** | Trop de requêtes (rate limiting) |
| **500** | Erreur serveur |

---
## 5. Analyse de données avec Pandas et Matplotlib

### 5.1 Créer un DataFrame depuis une API

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Récupérer 7 jours de températures horaires
url = "https://api.open-meteo.com/v1/forecast"
params = {
    "latitude": 49.90,
    "longitude": 2.30,
    "hourly": "temperature_2m,precipitation_probability",
    "forecast_days": 7
}

data = requests.get(url, params=params).json()

df = pd.DataFrame({
    "heure":       data["hourly"]["time"],
    "temperature": data["hourly"]["temperature_2m"],
    "precip_proba": data["hourly"]["precipitation_probability"]
})

df["heure"] = pd.to_datetime(df["heure"])
print(df.head())
print(f"\nShape : {df.shape}")

In [ ]:
# Statistiques descriptives
print(df[["temperature", "precip_proba"]].describe().round(1))

In [ ]:
# Visualisation
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

axes[0].plot(df["heure"], df["temperature"], color="tomato", linewidth=1.5)
axes[0].set_ylabel("Température (°C)")
axes[0].set_title("Prévisions météo 7 jours — Amiens")
axes[0].grid(True, alpha=0.3)

axes[1].fill_between(df["heure"], df["precip_proba"], color="steelblue", alpha=0.5)
axes[1].set_ylabel("Probabilité pluie (%)")
axes[1].set_xlabel("Date")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Sauvegarder les données
df.to_csv("meteo_amiens.csv", index=False)
print("Fichier sauvegardé : meteo_amiens.csv")

---
## 6. Lire et écrire des fichiers

### 6.1 CSV

In [ ]:
import pandas as pd

# Lire un CSV
# df = pd.read_csv("donnees.csv")
# df = pd.read_csv("donnees.csv", sep=";")          # séparateur point-virgule
# df = pd.read_csv("donnees.csv", encoding="utf-8") # encodage explicite

# Exemple : créer puis relire
ventes = pd.DataFrame({
    "mois": ["Jan", "Fév", "Mar", "Avr"],
    "ca":   [15000, 18000, 22000, 19000],
    "clients": [120, 145, 178, 155]
})
ventes.to_csv("ventes.csv", index=False)

# Relire et vérifier
df_ventes = pd.read_csv("ventes.csv")
print(df_ventes)
print(f"\nCA total : {df_ventes['ca'].sum():,} €")

In [ ]:
# JSON
import json

config = {
    "app": "DataAnalyser",
    "version": "1.0",
    "parametres": {"max_lignes": 10000, "timeout": 30}
}

# Écrire
with open("config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

# Lire
with open("config.json", "r", encoding="utf-8") as f:
    config_lu = json.load(f)

print(config_lu["parametres"]["timeout"])

In [ ]:
# Excel (nécessite openpyxl)
# uv add openpyxl

# Écrire en Excel
# df_ventes.to_excel("ventes.xlsx", sheet_name="Ventes 2024", index=False)

# Lire depuis Excel
# df = pd.read_excel("ventes.xlsx", sheet_name="Ventes 2024")

# Lire plusieurs feuilles
# feuilles = pd.read_excel("classeur.xlsx", sheet_name=None)  # dict feuille → DataFrame
print("(Décommenter pour tester avec un vrai fichier Excel)")

---
## 7. Organiser son code

Au-delà d'un notebook, on organise le code en modules Python.

In [ ]:
# Exemple : structure d'un projet d'analyse
# analyse_ventes/
#   data/
#     ventes.csv
#   output/
#     graphique.png
#     rapport.csv
#   utils.py        ← fonctions réutilisables
#   main.py         ← script principal
#   pyproject.toml    ← dépendances (géré par uv)

# Contenu de utils.py :
"""
import pandas as pd
import matplotlib.pyplot as plt

def charger_donnees(chemin):
    return pd.read_csv(chemin)

def calculer_stats(df, colonne):
    return {
        "moyenne": df[colonne].mean(),
        "max": df[colonne].max(),
        "min": df[colonne].min()
    }

def sauvegarder_graphique(fig, chemin):
    fig.savefig(chemin, dpi=150, bbox_inches="tight")
    print(f"Graphique sauvegardé : {chemin}")
"""

# Contenu de main.py :
"""
from utils import charger_donnees, calculer_stats, sauvegarder_graphique

df = charger_donnees("data/ventes.csv")
stats = calculer_stats(df, "ca")
print(stats)
"""

print("Structure de projet type affichée en commentaire")

---
## 8. Récapitulatif

```python
# Importer
import module                     # puis module.element
from module import element         # element directement
import module as alias             # alias court (np, pd, plt)

# Bibliothèque standard
import random, datetime, os, json

# Appel API
import requests
reponse = requests.get(url, params={"clé": "valeur"})
data = reponse.json()              # dict Python

# Pandas
df = pd.read_csv("fichier.csv")
df = pd.DataFrame({"col1": [...], "col2": [...]})
df.to_csv("sortie.csv", index=False)
```

| Tâche | Outil |
|-------|-------|
| Installer un paquet | `uv add nom` |
| Verrouiller les dépendances | `uv.lock` (automatique via `uv add`) |
| Appeler une API | `requests.get(url)` |
| Lire données tabulaires | `pd.read_csv()` / `pd.read_excel()` |
| Lire/écrire JSON | `json.load()` / `json.dump()` |

### Prochain CM
**CM06 — Python pratique & POO** : gestion d'erreurs (try/except), chemins de fichiers, et programmation orientée objet (classes, héritage).